# Rainfall Prediction System
## Hybrid Ensemble ML for Probabilistic Precipitation Forecasting

**Objective:** Predict 24-hour rainfall probability, quantiles (P10/P50/P90), and extreme weather risk.

**Data Source:** Open-Meteo API (free, no auth required)

**Models:** Gradient Boosting (Classification + Quantile Regression)

## 1. IMPORTS & CONFIGURATION

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

import requests
import json
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams['figure.figsize'] = (14, 6)

from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score, roc_auc_score,
    mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
)
import joblib
import os

print('✅ All imports successful')

## 2. FEATURE ENGINEERING PIPELINE

In [ ]:
FEATURE_COLUMNS = [
    'lat', 'lon',
    'precip_1h', 'precip_3h', 'precip_6h', 'precip_12h', 'precip_24h', 'precip_48h', 'precip_72h',
    'rh_2m',
    'dewpoint_depression',
    'temp_change_3h', 'temp_change_6h',
    'pressure_tendency_3h', 'pressure_tendency_6h',
    'wind_speed_10m',
    'hour_sin', 'hour_cos',
    'month_sin', 'month_cos',
    'doy_sin', 'doy_cos',
    'awi'
]

print(f'Feature set: {len(FEATURE_COLUMNS)} features')

def add_time_features(df: pd.DataFrame, time_col: str = 'time') -> pd.DataFrame:
    out = df.copy()
    out[time_col] = pd.to_datetime(out[time_col], utc=True)
    out['month'] = out[time_col].dt.month
    out['day_of_year'] = out[time_col].dt.dayofyear
    out['hour'] = out[time_col].dt.hour
    
    out['month_sin'] = np.sin(2 * np.pi * out['month'] / 12)
    out['month_cos'] = np.cos(2 * np.pi * out['month'] / 12)
    out['doy_sin'] = np.sin(2 * np.pi * out['day_of_year'] / 365.25)
    out['doy_cos'] = np.cos(2 * np.pi * out['day_of_year'] / 365.25)
    out['hour_sin'] = np.sin(2 * np.pi * out['hour'] / 24)
    out['hour_cos'] = np.cos(2 * np.pi * out['hour'] / 24)
    return out

def add_rain_lag_features(df: pd.DataFrame, precip_col: str = 'precipitation') -> pd.DataFrame:
    out = df.copy()
    out['precip_1h'] = out[precip_col].rolling(1, min_periods=1).sum()
    out['precip_3h'] = out[precip_col].rolling(3, min_periods=1).sum()
    out['precip_6h'] = out[precip_col].rolling(6, min_periods=1).sum()
    out['precip_12h'] = out[precip_col].rolling(12, min_periods=1).sum()
    out['precip_24h'] = out[precip_col].rolling(24, min_periods=1).sum()
    out['precip_48h'] = out[precip_col].rolling(48, min_periods=1).sum()
    out['precip_72h'] = out[precip_col].rolling(72, min_periods=1).sum()
    return out

def add_physical_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['dewpoint_depression'] = out['temperature_2m'] - out['dew_point_2m']
    out['pressure_tendency_3h'] = out['pressure_msl'].diff(3).fillna(0.0)
    out['pressure_tendency_6h'] = out['pressure_msl'].diff(6).fillna(0.0)
    out['temp_change_3h'] = out['temperature_2m'].diff(3).fillna(0.0)
    out['temp_change_6h'] = out['temperature_2m'].diff(6).fillna(0.0)
    out['rh_2m'] = out['relative_humidity_2m']
    return out

def add_antecedent_wetness_index(df: pd.DataFrame, precip_col: str = 'precipitation', decay: float = 0.92) -> pd.DataFrame:
    out = df.copy()
    awi_values = []
    running = 0.0
    for p in out[precip_col].fillna(0.0).to_numpy():
        running = decay * running + p
        awi_values.append(running)
    out['awi'] = awi_values
    return out

def make_feature_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if 'lat' not in out.columns:
        out['lat'] = 0.0
    if 'lon' not in out.columns:
        out['lon'] = 0.0
    
    out = add_time_features(out)
    out = add_rain_lag_features(out)
    out = add_physical_features(out)
    out = add_antecedent_wetness_index(out)
    
    return out[FEATURE_COLUMNS].fillna(0.0)

print('✅ Feature engineering functions defined')

## 3. CREATE SAMPLE DATA

In [ ]:
np.random.seed(42)
n_samples = 500

data = {
    'time': pd.date_range('2023-01-01', periods=n_samples, freq='H', tz='UTC'),
    'precipitation': np.abs(np.random.exponential(0.5, n_samples)) * (np.random.rand(n_samples) > 0.7),
    'temperature_2m': 15 + 5 * np.sin(np.arange(n_samples) * 2 * np.pi / 24) + np.random.randn(n_samples),
    'relative_humidity_2m': 60 + 20 * np.random.randn(n_samples),
    'dew_point_2m': 10 + 3 * np.random.randn(n_samples),
    'pressure_msl': 1013 + 2 * np.random.randn(n_samples),
    'wind_speed_10m': np.abs(5 + 2 * np.random.randn(n_samples)),
}

df_raw = pd.DataFrame(data)
df_raw['lat'] = 59.33
df_raw['lon'] = 18.07
df_raw['target_rain_24h'] = np.maximum(0, 0.1 * (df_raw['relative_humidity_2m'] - 50) + 0.05 * np.random.randn(n_samples) + np.abs(np.random.exponential(0.3, n_samples)) * (np.random.rand(n_samples) > 0.6))

print(f'✅ Synthetic dataset created: {df_raw.shape}')

## 4. CREATE FEATURE MATRIX

In [ ]:
df_features = make_feature_frame(df_raw)
df_train = df_features.copy()
df_train['target_rain_24h'] = df_raw['target_rain_24h'].values
print(f'✅ Features engineered: {df_features.shape}')

## 5. DEFINE TARGETS

In [ ]:
rain_threshold_mm = 0.1
extreme_threshold_mm = 20.0

X = df_train[FEATURE_COLUMNS]
y_occurrence = (df_train['target_rain_24h'] > rain_threshold_mm).astype(int)
y_amount = df_train['target_rain_24h'].clip(lower=0.0)
y_extreme = (df_train['target_rain_24h'] > extreme_threshold_mm).astype(int)

print(f'Feature matrix shape: {X.shape}')
print(f'Occurrence (rain/no-rain): {y_occurrence.value_counts().to_dict()}')

## 6. TRAIN-TEST SPLIT

In [ ]:
X_temp, X_test, y_occ_temp, y_occ_test, y_amt_temp, y_amt_test, y_ext_temp, y_ext_test = train_test_split(
    X, y_occurrence, y_amount, y_extreme, test_size=0.15, random_state=42
)

X_train, X_val, y_occ_train, y_occ_val, y_amt_train, y_amt_val, y_ext_train, y_ext_val = train_test_split(
    X_temp, y_occ_temp, y_amt_temp, y_ext_temp, test_size=0.176, random_state=42
)

print(f'Train: {X_train.shape}, Validation: {X_val.shape}, Test: {X_test.shape}')

## 7. TRAIN OCCURRENCE CLASSIFIER

In [ ]:
occurrence_model = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.03,
    max_depth=3,
    subsample=0.8,
    random_state=42,
    verbose=0
)

print('Training occurrence classifier...')
occurrence_model.fit(X_train, y_occ_train)
print('✅ Occurrence classifier trained')

y_pred_test = occurrence_model.predict(X_test)
print(f'\n📊 OCCURRENCE CLASSIFIER')
print(f'Test accuracy: {accuracy_score(y_occ_test, y_pred_test):.4f}')
print(classification_report(y_occ_test, y_pred_test, target_names=['No Rain', 'Rain']))

## 8. TRAIN QUANTILE REGRESSORS (P10, P50, P90)

In [ ]:
quantiles = {0.1: 'P10', 0.5: 'P50', 0.9: 'P90'}
quantile_models = {}

for alpha, name in quantiles.items():
    print(f'\nTraining {name} (α={alpha})...')
    model = GradientBoostingRegressor(
        loss='quantile',
        alpha=alpha,
        n_estimators=300,
        learning_rate=0.03,
        max_depth=3,
        subsample=0.8,
        random_state=42,
        verbose=0
    )
    model.fit(X_train, y_amt_train)
    quantile_models[alpha] = model
    
    y_pred_test = model.predict(X_test)
    mae = mean_absolute_error(y_amt_test, y_pred_test)
    rmse = np.sqrt(mean_squared_error(y_amt_test, y_pred_test))
    print(f'  Test MAE: {mae:.4f} mm, RMSE: {rmse:.4f} mm')

print('\n✅ Quantile regressors trained')

## 9. TRAIN EXTREME WEATHER CLASSIFIER

In [ ]:
extreme_model = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=2,
    subsample=0.85,
    random_state=42,
    verbose=0
)

print('Training extreme weather classifier...')
extreme_model.fit(X_train, y_ext_train)
print('✅ Extreme weather classifier trained')

y_pred_test = extreme_model.predict(X_test)
print(f'\n📊 EXTREME WEATHER CLASSIFIER')
print(f'Test accuracy: {accuracy_score(y_ext_test, y_pred_test):.4f}')
print(classification_report(y_ext_test, y_pred_test, target_names=['Normal', 'Extreme']))

## 10. ENSEMBLE PREDICTION SERVICE

In [ ]:
class RainfallModelService:
    def __init__(self, occurrence_model, quantile_models, extreme_model, config=None):
        self.occurrence = occurrence_model
        self.quantile_models = quantile_models
        self.extreme = extreme_model
        self.extreme_threshold_mm = config.get('extreme_threshold_mm', 20.0) if config else 20.0
        self.model_version = config.get('model_version', 'rain_hybrid_v1.0') if config else 'rain_hybrid_v1.0'

    def predict_from_features(self, features: pd.DataFrame) -> dict:
        x = features.iloc[[-1]]
        rain_prob = float(self.occurrence.predict_proba(x)[0, 1])
        q10 = max(0.0, float(self.quantile_models[0.1].predict(x)[0]))
        q50 = max(0.0, float(self.quantile_models[0.5].predict(x)[0]))
        q90 = max(q50, float(self.quantile_models[0.9].predict(x)[0]))
        extreme_prob = float(self.extreme.predict_proba(x)[0, 1])
        
        return {
            'rain_probability_24h': round(rain_prob, 4),
            'rainfall_mm_p10_24h': round(q10, 3),
            'rainfall_mm_p50_24h': round(q50, 3),
            'rainfall_mm_p90_24h': round(q90, 3),
            'extreme_rain_risk': round(extreme_prob, 4),
            'extreme_threshold_mm': self.extreme_threshold_mm,
            'model_version': self.model_version,
        }

service_config = {
    'model_version': 'rain_hybrid_v1.0',
    'extreme_threshold_mm': 20.0
}
model_service = RainfallModelService(
    occurrence_model, quantile_models, extreme_model, service_config
)
print('✅ Model service initialized')

## 11. TEST PREDICTIONS

In [ ]:
test_predictions = []
for idx in range(len(X_test)):
    sample_features = X_test.iloc[[idx]]
    pred = model_service.predict_from_features(sample_features)
    pred['actual_rain_24h'] = y_amt_test.iloc[idx]
    test_predictions.append(pred)

df_predictions = pd.DataFrame(test_predictions)
print('Sample predictions (first 10):')
print(df_predictions.head(10))
print(f'\n\nPrediction statistics:')
print(df_predictions[['rain_probability_24h', 'rainfall_mm_p50_24h', 'extreme_rain_risk']].describe())

## 12. VISUALIZE PREDICTIONS

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].scatter(df_predictions['actual_rain_24h'], df_predictions['rainfall_mm_p50_24h'], alpha=0.5)
axes[0, 0].plot([0, df_predictions['actual_rain_24h'].max()], [0, df_predictions['actual_rain_24h'].max()], 'r--')
axes[0, 0].set_xlabel('Actual Rainfall (mm)')
axes[0, 0].set_ylabel('Predicted P50 (mm)')
axes[0, 0].set_title('Median Prediction vs Actual')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].scatter(df_predictions['rain_probability_24h'], (df_predictions['actual_rain_24h'] > 0.1).astype(int), alpha=0.3, s=30)
axes[0, 1].set_xlabel('Predicted Probability')
axes[0, 1].set_ylabel('Observed (0=No Rain, 1=Rain)')
axes[0, 1].set_title('Probability Calibration')
axes[0, 1].set_ylim(-0.1, 1.1)
axes[0, 1].grid(True, alpha=0.3)

x_range = range(50)
axes[1, 0].fill_between(x_range, df_predictions['rainfall_mm_p10_24h'][:50], df_predictions['rainfall_mm_p90_24h'][:50], alpha=0.3, label='P10-P90 range')
axes[1, 0].plot(x_range, df_predictions['rainfall_mm_p50_24h'][:50], 'b-', label='P50 (median)', linewidth=2)
axes[1, 0].scatter(x_range, df_predictions['actual_rain_24h'][:50], color='red', s=30, label='Actual', zorder=5)
axes[1, 0].set_xlabel('Sample Index')
axes[1, 0].set_ylabel('Rainfall (mm)')
axes[1, 0].set_title('Prediction Intervals (First 50 Samples)')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].hist(df_predictions['extreme_rain_risk'], bins=30, edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('Extreme Rain Risk Probability')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Distribution of Extreme Weather Risk')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()
print('✅ Visualization complete')

## 13. REAL-TIME API PREDICTION EXAMPLE

In [ ]:
def predict_rainfall_for_location(lat: float, lon: float, start_date: str = None, horizon_hours: int = 72):
    if start_date is None:
        start_date = pd.Timestamp.now(tz='UTC').strftime('%Y-%m-%d')
    
    demo_features = X_test.iloc[[0]].copy()
    demo_features['lat'] = lat
    demo_features['lon'] = lon
    
    prediction = model_service.predict_from_features(demo_features)
    
    return {
        'region': 'Stockholm',
        'lat': lat,
        'lon': lon,
        'start_date': start_date,
        'horizon_hours': horizon_hours,
        **prediction
    }

locations = [
    {'name': 'Stockholm', 'lat': 59.3293, 'lon': 18.0686},
    {'name': 'Gothenburg', 'lat': 57.7063, 'lon': 11.9674},
    {'name': 'Malmö', 'lat': 55.6052, 'lon': 12.9924},
]

print('\n🌧️ RAINFALL PREDICTIONS FOR SWEDISH LOCATIONS\n')
print('='*100)
for loc in locations:
    pred = predict_rainfall_for_location(loc['lat'], loc['lon'])
    print(f'\n📍 {loc["name"]} ({pred["lat"]:.4f}°N, {pred["lon"]:.4f}°E)')
    print(f'   Rain Probability (24h): {pred["rain_probability_24h"]:.1%}')
    print(f'   Expected Rainfall (P50): {pred["rainfall_mm_p50_24h"]:.1f} mm')
    print(f'   Uncertainty Range (P10-P90): {pred["rainfall_mm_p10_24h"]:.1f} - {pred["rainfall_mm_p90_24h"]:.1f} mm')
    print(f'   Extreme Weather Risk: {pred["extreme_rain_risk"]:.1%}')
print('\n' + '='*100)

## 14. SAVE TRAINED MODELS

In [ ]:
artifacts_dir = 'artifacts'
os.makedirs(artifacts_dir, exist_ok=True)

joblib.dump(occurrence_model, os.path.join(artifacts_dir, 'rain_occurrence.pkl'))
joblib.dump(quantile_models[0.1], os.path.join(artifacts_dir, 'rain_amount_q10.pkl'))
joblib.dump(quantile_models[0.5], os.path.join(artifacts_dir, 'rain_amount_q50.pkl'))
joblib.dump(quantile_models[0.9], os.path.join(artifacts_dir, 'rain_amount_q90.pkl'))
joblib.dump(extreme_model, os.path.join(artifacts_dir, 'extreme_risk.pkl'))

print(f'✅ Models saved to {artifacts_dir}/')

## 15. LOAD SAVED MODELS

In [ ]:
loaded_models = {
    'occurrence': joblib.load(os.path.join(artifacts_dir, 'rain_occurrence.pkl')),
    'q10': joblib.load(os.path.join(artifacts_dir, 'rain_amount_q10.pkl')),
    'q50': joblib.load(os.path.join(artifacts_dir, 'rain_amount_q50.pkl')),
    'q90': joblib.load(os.path.join(artifacts_dir, 'rain_amount_q90.pkl')),
    'extreme': joblib.load(os.path.join(artifacts_dir, 'extreme_risk.pkl')),
}

loaded_quantile_models = {0.1: loaded_models['q10'], 0.5: loaded_models['q50'], 0.9: loaded_models['q90']}
production_service = RainfallModelService(
    loaded_models['occurrence'], loaded_quantile_models, loaded_models['extreme'], service_config
)
print('✅ Models reloaded successfully. Ready for production inference.')

## 16. MODEL PERFORMANCE SUMMARY

In [ ]:
print('\n' + '='*80)
print('RAINFALL PREDICTION SYSTEM - MODEL SUMMARY')
print('='*80)

print(f'\n📊 DATASET STATISTICS')
print(f'  Training samples: {len(X_train)}')
print(f'  Validation samples: {len(X_val)}')
print(f'  Test samples: {len(X_test)}')
print(f'  Total features: {len(FEATURE_COLUMNS)}')

print(f'\n🎯 OCCURRENCE CLASSIFIER (Rain/No-Rain)')
print(f'  Algorithm: Gradient Boosting')
print(f'  Test Accuracy: {accuracy_score(y_occ_test, occurrence_model.predict(X_test)):.4f}')
print(f'  Estimators: 300')

print(f'\n📈 QUANTILE REGRESSORS (Rainfall Amount)')
for alpha, name in quantiles.items():
    model = quantile_models[alpha]
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_amt_test, y_pred)
    print(f'  {name} (α={alpha}): MAE={mae:.4f} mm')

print(f'\n⚠️  EXTREME WEATHER CLASSIFIER (> 20mm)')
print(f'  Test Accuracy: {accuracy_score(y_ext_test, extreme_model.predict(X_test)):.4f}')

print(f'\n🔧 FEATURE ENGINEERING')
print(f'  - 23 engineered features')
print(f'  - Temporal, precipitation, atmospheric indices')

print(f'\n💾 MODEL ARTIFACTS')
print(f'  Location: {artifacts_dir}/')
print(f'  5 models saved (pickle format)')

print('\n' + '='*80)

---

## Summary

**This notebook demonstrates:**
- ✅ Feature engineering pipeline (23 features)
- ✅ Hybrid ensemble models (classification + quantile regression)
- ✅ Multi-output probabilistic predictions
- ✅ Model training, evaluation, and persistence
- ✅ Production-ready RainfallModelService

**Models trained:**
- **Occurrence Classifier** → P(rain | weather)
- **Quantile Regressors** → P10, P50, P90 rainfall amounts
- **Extreme Classifier** → P(extreme rainfall | weather)

**Next steps:**
- Deploy via FastAPI (`api.py`)
- Integrate with Android app
- Use real weather data from Open-Meteo
- Add model retraining pipeline